In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def ode_rk4_step(f, y, x, h):
    """Perform a single step h using 4th order Runge-Kutta method.

    Args:
    f: the function that defines the ODE.
    x: the value of the dependent variable at the present step.
    t: the present value of the time variable.
    h: the time step

    Returns:
    xnew: the value of the dependent variable at the step t+h
    """
    k1 = h * f(y,x)
    k2 = h * f(y + k1/2., x + h /2.)
    k3 = h * f(y + k2/2., x + h /2.)
    k4 = h * f(y + k3, x + h)
    return y + (k1 + 2. * k2 + 2. * k3 + k4) / 6.

def ode_rk4_multi(f, y0, x0, h, nsteps):
    """Multi-dimensional version of the RK4 method.
    """

    x = np.zeros(nsteps + 1)
    y = np.zeros((len(x), len(y0)))
    x[0] = x0
    y[0] = y0
    for i in range(0, nsteps):
        x[i + 1] = x[i] + h
        y[i + 1] = ode_rk4_step(f, y[i], x[i], h)
    return x,y

def f(xin,x):
    y = xin[0]
    z = xin[1]
    return np.array([z,-2*z/x - 2*y/x**2 + np.sin(np.log(x))/x**2])

# Initial and final times
x0 = 1.
xend = 2.
# Number of RK4 steps
Nrk4 = 10
hrk4 = (xend - x0) / Nrk4
sol1 = ode_rk4_multi(f, [1, 0], x0, hrk4, Nrk4)[1]
sol2 = ode_rk4_multi(f, [1, 1], x0, hrk4, Nrk4)[1]

lam = (2 - sol2[:,0][-1])/(sol1[:,0][-1]-sol2[:,0][-1])

ans = lam * sol1[:,0] + (1.0 - lam)*sol2[:,0]
plt.plot(ans)
print(ans)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def f(xin,x):
    y = xin[0]
    yp = xin[1]
    return np.array([yp,(32. + 2.*x**3 - y*yp)/8.])

def fy(yp):
    return -yp/8
def fyp(y):
    return -y/8

def g(xin,fy,fyp,x):
    return np.array([xin[1],fy*xin[0] + fyp*xin[1]])



# Initial and final times
x0 = 1.
xend = 3.
alpha = 17
beta = 43/3
Nrk4 = 20
h = (xend - x0) / Nrk4
iteration_counter = 0
guess = (beta - alpha)/(xend - x0)
y0 = np.array([17,guess])
u0 = [0,1]

while iteration_counter<50:
    x = np.zeros(Nrk4+1)
    y = np.zeros((Nrk4+1, 2))
    u = np.zeros((Nrk4+1, 2))
    y[0] = y0
    u[0] = u0
    z1 = 0
    z2 = 1

    for i in range(Nrk4):
        x[i+1] = x[i] + h

        k1 = h*f(y[i],x[i])
        k2 = h*f(y[i]+k1/2, x[i]+h/2)
        k3 = h*f(y[i]+k2/2, x[i]+h/2)
        k4 = h*f(y[i]+k3, x[i]+h)

        y[i+1] = y[i] + (k1 + k4 + 2.*(k2 + k3))/6.

        k1p = h*g(u[i],x[i], y[i][1],y[i][0])
        k2p = h*g(u[i]+k1p/2, x[i]+h/2,  y[i][1],y[i][0])
        k3p = h*g(u[i]+k2p/2, x[i]+h/2,  y[i][1],y[i][0])
        k4p = h*g(u[i]+k3p, x[i]+h,  y[i][1],y[i][0])

        u[i+1] = u[i] + (k1p + k4p + 2*(k2p + k3p))/6

        if abs(y[0][-1]-beta)<1.e-5:
            exit
        else:
            guess=guess- (y[0][-1] - beta)/u[0][-1]
    iteration_counter +=1

print(y)